## 7. Verification Summary

Expected Success Indicators ✅

- [ ] Phase 1 Security
  - SQL injection queries blocked (400 error)
  - DROP TABLE queries blocked (400 error)

- [ ] Phase 2 Performance
  - Cache hit latency < 10ms (vs 40ms+ on miss)
  - Budget endpoint returns daily_budget=50000
  - Cost estimation returns numeric values
  - Redis cache keys present after queries

- [ ] Data Integrity
  - Audit logs capture all queries
  - Role-based PII masking applied
  - Cache invalidates on data writes

In [ ]:
print("📦 Redis Cache Verification\n")

# Check Redis cache keys
print("1️⃣ Redis Cache Keys:")
result = subprocess.run(
    ['docker-compose', 'exec', '-T', 'redis', 'redis-cli', 'KEYS', 'siqg:cache:*'],
    cwd='/mnt/c/Users/mmuaz/Desktop/Projects/siqg',
    capture_output=True,
    text=True
)

if result.returncode == 0 and result.stdout.strip():
    keys = result.stdout.strip().split('\n')
    print(f"   ✅ Found {len(keys)} cache keys:")
    for key in keys[:5]:  # Show first 5
        print(f"      - {key}")
    if len(keys) > 5:
        print(f"      ... and {len(keys) - 5} more")
else:
    print(f"   ⚠️ No cache keys found yet (normal on first run)")

# Check cache tags (for invalidation)
print("\n2️⃣ Cache Tags (table invalidation):")
result = subprocess.run(
    ['docker-compose', 'exec', '-T', 'redis', 'redis-cli', 'KEYS', 'siqg:cache_tags:*'],
    cwd='/mnt/c/Users/mmuaz/Desktop/Projects/siqg',
    capture_output=True,
    text=True
)

if result.returncode == 0 and result.stdout.strip():
    keys = result.stdout.strip().split('\n')
    print(f"   ✅ Found {len(keys)} table tags:")
    for key in keys[:5]:
        print(f"      - {key}")
else:
    print(f"   ⚠️ No cache tags found yet")

# Check budget keys
print("\n3️⃣ Budget Tracking Keys:")
result = subprocess.run(
    ['docker-compose', 'exec', '-T', 'redis', 'redis-cli', 'KEYS', 'siqg:budget:*'],
    cwd='/mnt/c/Users/mmuaz/Desktop/Projects/siqg',
    capture_output=True,
    text=True
)

if result.returncode == 0 and result.stdout.strip():
    keys = result.stdout.strip().split('\n')
    print(f"   ✅ Found {len(keys)} budget keys:")
    for key in keys[:5]:
        # Get the value
        val_result = subprocess.run(
            ['docker-compose', 'exec', '-T', 'redis', 'redis-cli', 'GET', key],
            cwd='/mnt/c/Users/mmuaz/Desktop/Projects/siqg',
            capture_output=True,
            text=True
        )
        value = val_result.stdout.strip()
        print(f"      - {key}: {value}")
else:
    print(f"   ⚠️ No budget keys found yet")

print("\n" + "="*60)

## 6. Verify Redis Cache Backend

Check Redis keys and cache structure directly.

In [ ]:
print("⚡ Phase 2 Performance Tests\n")

test_query = "SELECT 42 as answer"

# Test 1: Cache Miss (First Execution)
print("1️⃣ Testing Cache Miss (first query execution)...")
start = time.time()
response = requests.post(
    f"{BASE_URL}/api/v1/query/execute",
    json={"query": test_query},
    headers=HEADERS,
    timeout=5
)
latency_ms = (time.time() - start) * 1000

if response.status_code == 200:
    data = response.json()
    cached = data.get("cached", False)
    cost = data.get("cost", "N/A")
    actual_latency = data.get("latency_ms", "N/A")

    print(f"   ✅ Query executed")
    print(f"      - Cached: {cached}")
    print(f"      - Latency (reported): {actual_latency}ms")
    print(f"      - Cost: {cost}")
else:
    print(f"   ❌ Error {response.status_code}: {response.text}")

# Test 2: Cache Hit (Second Execution - Identical Query)
print("\n2️⃣ Testing Cache Hit (identical query)...")
time.sleep(0.5)
start = time.time()
response = requests.post(
    f"{BASE_URL}/api/v1/query/execute",
    json={"query": test_query},
    headers=HEADERS,
    timeout=5
)
latency_ms = (time.time() - start) * 1000

if response.status_code == 200:
    data = response.json()
    cached = data.get("cached", False)
    actual_latency = data.get("latency_ms", "N/A")

    print(f"   ✅ Query returned")
    print(f"      - Cached: {cached}")
    print(f"      - Latency (reported): {actual_latency}ms")

    if cached:
        print(f"      ✅ CACHE HIT VERIFIED - Should be <10ms vs 40ms+ for fresh execution")
    else:
        print(f"      ⚠️ Not cached - may need to wait for TTL or check Redis")
else:
    print(f"   ❌ Error {response.status_code}: {response.text}")

# Test 3: Budget Tracking
print("\n3️⃣ Testing Budget Status (Phase 2)...")
response = requests.get(
    f"{BASE_URL}/api/v1/query/budget",
    headers=HEADERS,
    timeout=5
)

if response.status_code == 200:
    budget = response.json()
    print(f"   ✅ Budget endpoint working")
    print(f"      - Daily Budget: {budget.get('daily_budget', 'N/A')} units")
    print(f"      - Current Usage: {budget.get('current_usage', 'N/A')} units")
    print(f"      - Remaining: {budget.get('remaining', 'N/A')} units")
    print(f"      - Resets At: {budget.get('resets_at', 'N/A')}")
else:
    print(f"   ❌ Error {response.status_code}: {response.text}")

# Test 4: Cost Estimation
print("\n4️⃣ Testing Cost Estimation...")
response = requests.post(
    f"{BASE_URL}/api/v1/query/execute",
    json={"query": "SELECT 1"},
    headers=HEADERS,
    timeout=5
)

if response.status_code == 200:
    data = response.json()
    cost = data.get("cost", None)
    if cost is not None and cost > 0:
        print(f"   ✅ Cost estimation working")
        print(f"      - Estimated Cost: {cost}")
    else:
        print(f"   ⚠️ Cost field missing or zero: {cost}")
else:
    print(f"   ❌ Error {response.status_code}")

print("\n" + "="*60)

## 5. Test Phase 2 Performance Features

Verify caching, cost estimation, budget tracking, and auto-limit injection.

In [ ]:
print("🔒 Phase 1 Security Tests\n")

# Test 1: SQL Injection Detection
print("1️⃣ Testing SQL Injection Detection...")
sql_injection = "SELECT * FROM users WHERE id = 1 OR 1=1"
response = requests.post(
    f"{BASE_URL}/api/v1/query/execute",
    json={"query": sql_injection},
    headers=HEADERS,
    timeout=5
)

if response.status_code == 400:
    detail = response.json().get("detail", "")
    if "injection" in detail.lower():
        print(f"   ✅ BLOCKED: {detail}")
    else:
        print(f"   ⚠️ Got 400 but unclear reason: {detail}")
else:
    print(f"   ❌ Expected 400, got {response.status_code}: {response.text}")

# Test 2: DROP TABLE Blocking
print("\n2️⃣ Testing DROP TABLE Blocking...")
drop_query = "DROP TABLE users"
response = requests.post(
    f"{BASE_URL}/api/v1/query/execute",
    json={"query": drop_query},
    headers=HEADERS,
    timeout=5
)

if response.status_code == 400:
    detail = response.json().get("detail", "")
    print(f"   ✅ BLOCKED: {detail}")
else:
    print(f"   ❌ Expected 400, got {response.status_code}: {response.text}")

print("\n" + "="*60)

## 4. Test Phase 1 Security (Validation)

Verify SQL injection and DROP TABLE blocking still working.

In [ ]:
# Try login first (user may already exist)
login_response = requests.post(
    f"{BASE_URL}/api/v1/auth/login",
    json={"username": "testuser", "password": "testpass123"},
    timeout=5
)

token = None
if login_response.status_code == 200:
    token = login_response.json().get("access_token")
    print(f"✅ Login successful")
else:
    # If login failed, try to register
    reg_response = requests.post(
        f"{BASE_URL}/api/v1/auth/register",
        json={
            "username": "testuser",
            "email": "testuser@example.com",
            "password": "testpass123"
        },
        timeout=5
    )
    if reg_response.status_code == 200:
        token = reg_response.json().get("access_token")
        print(f"✅ Registration successful")
    else:
        print(f"❌ Authentication failed: {reg_response.text}")

if token:
    print(f"✅ Token: {token[:30]}...")
else:
    print("❌ Failed to get authentication token")

# Store token for subsequent requests
HEADERS = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
} if token else {"Content-Type": "application/json"}

## 3. User Authentication

Set up test user and get authentication token.

In [ ]:
import subprocess
import json

# Check Docker services status
result = subprocess.run(
    ['docker-compose', 'ps', '--format', 'json'],
    cwd='/mnt/c/Users/mmuaz/Desktop/Projects/siqg',
    capture_output=True,
    text=True
)

if result.returncode == 0:
    services = json.loads(result.stdout)
    print("📦 Docker Services Status:")
    print("-" * 60)
    for service in services:
        status_icon = "✅" if "Up" in service["State"] else "❌"
        print(f"{status_icon} {service['Service']:<15} | {service['State']}")
    print("-" * 60)
else:
    print("❌ Failed to get Docker status:", result.stderr)

# Check Gateway connectivity
try:
    response = requests.get(f"{BASE_URL}/docs", timeout=2)
    print(f"✅ Gateway is accessible at {BASE_URL}")
except:
    print(f"❌ Gateway is NOT accessible at {BASE_URL}")

## 2. Connect to Docker Services

Verify that all services (PostgreSQL, Redis, Gateway) are running and healthy.

In [ ]:
import requests
import json
import time
import subprocess
from datetime import datetime
import re

# Configuration
BASE_URL = "http://localhost:8000"
VERBOSE = True  # Set to False to reduce output

def log(msg):
    if VERBOSE:
        print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}")

print("✅ Libraries imported successfully")

## 1. Import Required Libraries

This section imports libraries for HTTP requests, database connections, and Docker interaction.

# SIQG Phase 2 Verification Tests

This notebook validates all Phase 2 performance middleware components:
- Query fingerprinting & caching (Redis)
- Cost estimation (PostgreSQL EXPLAIN)
- Daily budget tracking & enforcement
- Auto-LIMIT injection for unbounded queries
- Cache invalidation on data writes

**Quick Start**: Run cells 1-6 to verify Phase 2 is working ✅